# Datamine INDEST Process Example

This notebook demonstrates how to configure and run the **`indest`** process wrapper in `dmstudio`.

## Process Description

## INDEST

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The INDEST process uses the Indicator Estimation (IE) method to estimate grades into a block model using the cumulative distribution function (CDF) of indicator transformed sample grades

To operate, INDEST needs a series of threshold values between the smallest and largest grade values. These threshold values, referred to as cutoffs, are used to numerically build the CDF of each block in the model. For each cutoff, data in the search volume are transformed into 0s and 1s: 1s if the data are greater than the threshold, and 0s if they are less than or equal. It then estimates the probability that the block grade is greater than the threshold value, using one of the standard estimation methods. This is usually kriging, but INDEST allows other methods such as Nearest Neighbour or Inverse Power of Distance to be used. Performing this operation for each cutoff across the range of the sample data approximates the CDF for the model cell. After the CDF is built, it is post processed to calculate the indicator estimated grade.

INDESTuses the ESTIMAprocess to do the estimation for each cutoff. For further details of ESTIMA.

If you are using Indicator Kriging (IK) then you must already have calculated a variogram for each cutoff, and stored the models in the Variogram Model file. The[VGRAM](<vgram.md>)process has specific features for calculating indicator variograms.

For each cutoff INDESTcalculates the following data which can optionally be stored in the **Output Model** file:

* The proportion (fraction) of the model subcell which is above cutoff.

* The grade of the proportion of the subcell which lies above cutoff.

The main output from INDESTis the grade above a cutoff of zero, ie the indicator estimated grade of the total subcell.

### Estimation Parameter File

In order to useINDESTyou must specify one record in the Estimation Parameter File &**ESTPARM** for each cutoff. This requires the numeric field **CUTOFF** to be included in the file.

The * **VALUE_IN** field is the grade in the input sample &**IN** file to which the cutoffs are applied. Note that this is different from the use of the * **VALUE_IN** field when using ESTIMATE for Indicator Estimation.

If the * **VALUE_OU** field is not specified then the * **VALUE_IN** field will be created in the output model file to hold the indicator estimated grade. If a * **VALUE_OU** field is specified in the first record of the Estimation Parameter File, then this value will be used for the indicator estimated grade in the output model file

You can only estimate one set of indicators in a single run. In other words all the **VALUE_IN** fields must be the same. Also when using **INDEST** you cannot estimate a grade using non indicator methods in the same run.

If you are using zone control then you must explicitly specify all combinations of zone field(s) in the Estimation Parameter File. You cannot use the option that is available in ESTIMA that allows you to specify an absent data zone field value that then applies to all zones that are not explicitly identified in&ESTPARM.

If you are using zone control then you may use different cutoffs for different zones. However the PRABn and GRABn fields in the output model file must then be handled very carefully! The maximum number of cutoff values is 24.

Fields for indicator estimation:

* **BINGRADE** : Used when **GRMETHOD** =4 to set the grade below the cutoff

* **ABVGRADE** : Sets the grade above the cutoff (only used for the top bin)

### Grade Above Cutoff

The calculation of the grade above each cutoff requires that the average grade between each successive pair of cutoffs be specified. For example if cutoff grades of 2, 5, 6.5 and 9.5g/t are selected then average grades are required for the ranges:

From  |  To
---|---
0 |  2
2 |  5
5 |  6.5
6.5 |  9.5
9.5 |  ∞

Four methods are available to specify the average grade for each range, controlled by parameter @**GRMETHOD** :

GRMETHOD  |  Description
---|---
1 |  Average of minimum and maximum cutoff values. The grade above the highest cutoff is calculated as the highest cutoff plus half the difference between the highest and second highest cutoffs.
2 |  Calculated by INDEST from the grades of the samples in the &**IN** file.
3 |  Calculated by INDEST from the grades of the samples in the &**IN** file. However for the top bin (above the highest cutoff) the median grade is calculated.
4 |  Values are specified by the user, using the **BINGRADE** and **ABVGRADE** fields in the &**ESTPARM** file. The **BINGRADE** field contains the grade below the cutoff and the **ABVGRADE** field the grade above the cutoff. The ABVGRADE field is therefore only used for the top bin.

**GRMETHOD** of 4 is illustrated in the following table:

## CUTOFF  | BINGRADE |  ABVGRADE

---|---|---
2 |  1.3 |  -
5 |  3.6 |  -
6.5 |  5.7 |
9.5 |  7.8 |  11.1

The values used by INDEST can be recorded by specifying an output &**AVGRADES** file.

### Indicators

Indicator values are calculated for each sample in the input sample &**IN** file for each cutoff. An indicator takes the value:

0 ‑ the grade is less than or equal to the cutoff.

1 ‑ the grade is above the cutoff.

The indicator values can be saved by specifying an output &**INDICATE** file.

###  Output Model

Fields **PRAB1 ... PRAB** n will be created in the Output Model file to store the proportion of the subcell above each cutoff. These are calculated directly by ESTIMA. Then the fields **GRAB1 ... GRAB** n are calculated during the post-processing to store the corresponding grade above each cutoff. The grade above cutoff values (**GRABn**) are calculated from the proportion and average grade between each pair of cutoffs. For example:

Cutoff Number  |  Cutoff  |  PRABn  |  Calculation
---|---|---|---
4 |  9.5 |  0.1 |  GRAB4= 11.1 (This figure is taken directly from the ABVGRADE field)
3 |  6.5 |  0.3 |  GRAB3= {0.1x11.1 + (0.3‑0.1)x7.8} / 0.3 = 8.9
2 |  5 |  0.6 |  GRAB2= {0.1x11.1 + (0.3‑0.1)x7.8 + (0.6‑0.3)x5.7} / 0.6 = 7.3
1 |  2 |  0.85 |  GRAB1= {0.1x11.1 + (0.3‑0.1)x7.8 + (0.6‑0.3)x5.7 + (0.85‑0.6)x3.6} / 0.85 = 6.21
0 |  0 |  1 |  Indicator Grade=0.1x11.1 + (0.3‑0.1)x7.8 + (0.6‑0.3)x5.7 \+ (0.85‑0.6)x3.6 + (1.0‑0.85)x1.3 = 5.48

The **PRABn** and **GRABn** fields will be stored in the output &**MODEL** file if parameter @PGFIELDS is set to 1.

### Order Relation

One of the main drawbacks of the indicator estimation method is the Order Relation Problem. This will occur if the proportion of the subcell above cutoff n is estimated to be less than the proportion above cutoff n+1. ie PRAB(n) < PRAB(n+1). Three options are available to correct for this, controlled by parameter ORDER:

=1: Downwards.
=2: Upwards.
=3: Average of methods 1 and 2.

The recommended method (and default) is 3.

### Input Files:

* **proto** (Block_Model_File):
  Input model prototype. This is a standard block model file containing the 13 compulsory
  fields. It may also contain the rotated model fields. If it includes cells then it must
  be sorted on IJK.
  Required=Yes

* **in** (Undefined):
  Input sample data. This must contain X,Y and Z fields and at least one grade field.
  Required=Yes

* **srcparm** (Undefined):
  Search volume parameter file. This contains 24 compulsory fields defining the search
  volume and the number of samples needed for grade estimation. More than one search
  volume may be defined. All fields are numeric:
  Required=Yes

* **estparm** (Undefined):
  Estimation parameter file. Each record in the file describes an estimation method and
  its associated parameters. The fields are dependent on the estimation methods selected.
  All fields are optional except for **VALUE_IN** , **SREFNUM** and **CUTOFF**. General

* **fields**:
  Required=Yes

* **vmodparm** (Variogram \- Model):
  Variogram model parameter file. Each record in this file defines a variogram model type
  and its parameters. Only the VREFNUM field is compulsory.
  Required=No

### Output Files:

* **model** (Block Model File):
  Output model containing estimated grades, variance etc.
  Required=No

* **avgrades** (Undefined):
  Output file containing cutoff grade ranges and average grade used for each range. It
  will include zone field(s), if any, plus the following fields:
  Required=No

* **indicate** (Undefined):
  Output indicator file. This is a copy of the sample input IN file, but also includes
  the 0/1 indicator values for each cutoff
  Required=No

* **sampout** (Undefined):
  Output sample file containing details of weights for each sample for each cell
  estimated.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate of sample data in IN file. If not specified, then X is assumed.
  Default=Undefined
  Required=No

* **y** (Numeric : IN):
  Y coordinate of sample data in IN file. If not specified, then Y is assumed.
  Default=Undefined
  Required=No

* **z** (Numeric : IN):
  Z coordinate of sample data in IN file. If not specified, then Z is assumed.
  Default=Undefined
  Required=No

* **zone1_f** (Any : IN):
  First field for zonal control.
  Default=Undefined
  Required=No

* **zone2_f** (Any : IN):
  Second field for zonal control.
  Default=Undefined
  Required=No

* **key** (Numeric : IN):
  Key field used to specify the field limiting the number of samples for estimation. The
  field must exist in the IN file.
  Default=Undefined
  Required=No

* **length_f** (Numeric : IN):
  Field used for length weighting in IPD. The field must exist in the IN file.
  Default=Undefined
  Required=No

* **dens_f** (Numeric : IN):
  Field used for density weighting in IPD. The field must exist in the IN file.
  Default=Undefined
  Required=No

### Parameters:

* **discmeth**:
  Cell discretisation method:
  Range=
  Values=
  Default=
  Required=No

* **xpoints**:
  Number of discretisation points in X. (1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ypoints**:
  Number of discretisation points in Y. (1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **zpoints**:
  Number of discretisation points in Z. (1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **xdspace**:
  Distance between discretisation points in X if DISCMETH=2. The default gives just one
  point.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ydspace**:
  Distance between discretisation points in Y if DISCMETH=2. The default gives just one
  point.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **zdspace**:
  Distance between discretisation points in Z if DISCMETH=2. The default gives just one
  point.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **parent**:
  Flag to control parent cell estimation:
  Range=
  Values=
  Default=
  Required=No

* **mindisc**:
  Minimum number of discretisation points. Only used if PARENT=2. The default is (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **copyval**:
  Flag controlling copying of values from PROTO to MODEL if there is insufficient data to
  estimate them:
  Range=
  Values=
  Default=
  Required=No

* **fvaltype**:
  Flag for cell size approximation for F values:
  Range=
  Values=
  Default=
  Required=No

* **fstep**:
  Step size for cell approximation. This is only used if **FVALTYPE** =2.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xmin**:
  Minimum X value for model updating. The default is the X model origin.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xmax**:
  Maximum X value for model updating. The default is the maximum X value for **PROTO**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymin**:
  Minimum Y value for model updating. The default is the Y model origin.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **ymax**:
  Maximum Y value for model updating. The default is the maximum Y value for **PROTO**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **zmin**:
  Minimum Z value for model updating. The default is the Z model origin.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **zmax**:
  Maximum Z value for model updating. The default is the maximum Z value for **PROTO**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **xsubcell**:
  Number of subcells per parent cell in X if **PROTO** is empty. The default is (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **ysubcell**:
  Number of subcells per parent cell in Y if **PROTO** is empty. The default is (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **zsubcell**:
  Number of subcells per parent cell in Z if **PROTO** is empty. The default is (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **order**:
  Order relation correction method:
  Range=
  Values=
  Default=
  Required=No

* **grmethod**:
  Method for specifying average grade within each cutoff range:
  Range=
  Values=
  Default=
  Required=No

* **pgfields**:
  Flag to select whether the proportion above cutoff fields (PRABn) and the grade above
  cutoff fields (GRABn) should be included in the output MODEL file:
  Range=
  Values=
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('indest')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute indest
print("Running indest...")
dm_cmd.indest(
    proto_i='t_mod1',  # required
    in_i='t_assays',  # required
    srcparm_i='t_spar',  # required
    estparm_i='t_epar',  # required
    order_p='optional',  # required
    # vmodparm_i='t_vpar',  # optional
    # model_o='t_indest_out',  # optional
    # avgrades_o=['t_indest_out'],  # optional
    # indicate_o='t_indest_out',  # optional
    # sampout_o='t_indest_out',  # optional
    # x_f='X',  # optional
    # y_f='Y',  # optional
    # z_f='Z',  # optional
    # zone1_f_f='optional',  # optional
    # zone2_f_f='optional',  # optional
    # key_f=['BHID'],  # optional
    # length_f_f='optional',  # optional
    # dens_f_f='optional',  # optional
    # discmeth_p='optional',  # optional
    # xpoints_p=1,  # optional
    # ypoints_p=1,  # optional
    # zpoints_p=1,  # optional
    # xdspace_p='optional',  # optional
    # ydspace_p='optional',  # optional
    # zdspace_p='optional',  # optional
    # parent_p='optional',  # optional
    # mindisc_p=1,  # optional
    # copyval_p='optional',  # optional
    # fvaltype_p='optional',  # optional
    # fstep_p='optional',  # optional
    # xmin_p='optional',  # optional
    # xmax_p='optional',  # optional
    # ymin_p='optional',  # optional
    # ymax_p='optional',  # optional
    # zmin_p='optional',  # optional
    # zmax_p='optional',  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # grmethod_p='optional',  # optional
    # pgfields_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("indest execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_indest_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")